# TimeSformer Finetuning Pipeline

Pipeline to train the TimeSformer model using the `cineca` branch and the Hugging Face dataset.

✅ Compatible with **Google Colab** and **Kaggle**.

---
### Kaggle setup (do this before running):
1. Go to **Add-ons → Secrets** and add a secret named `HF_TOKEN` with your Hugging Face access token.
2. Enable **Internet** in the notebook settings (right panel).
3. Enable **GPU** accelerator (P100 or T4).

### Colab setup:
1. Select **Runtime → Change runtime type → GPU**.
2. Run the notebook — you will be prompted for your HF token in cell 4.

## 1. Detect Environment & Set Paths

In [ ]:
import os
import sys

# ── Environment detection ──────────────────────────────────────────────────────
ON_KAGGLE = os.path.exists('/kaggle')
ON_COLAB  = 'google.colab' in sys.modules or os.path.exists('/content')

if ON_KAGGLE:
    PLATFORM     = 'kaggle'
    WORKING_DIR  = '/kaggle/working'
    REPO_DIR     = '/kaggle/working/Driving_Distraction_Detection'
    HF_CACHE_DIR = '/kaggle/working/hf_cache'
    # On Kaggle, outputs are saved in /kaggle/working (persisted after session)
    OUTPUT_DIR   = '/kaggle/working/timesformer_outputs'
elif ON_COLAB:
    PLATFORM     = 'colab'
    WORKING_DIR  = '/content'
    REPO_DIR     = '/content/Driving_Distraction_Detection'
    HF_CACHE_DIR = '/content/hf_cache'
    # Will be updated to Drive path in cell 2 if Drive is mounted
    OUTPUT_DIR   = '/content/drive/MyDrive/Driving_Distraction_Outputs'
else:
    PLATFORM     = 'local'
    WORKING_DIR  = os.getcwd()
    REPO_DIR     = os.getcwd()
    HF_CACHE_DIR = './hf_cache'
    OUTPUT_DIR   = './timesformer_outputs'

print(f'Platform   : {PLATFORM}')
print(f'Working dir: {WORKING_DIR}')
print(f'Repo dir   : {REPO_DIR}')
print(f'HF cache   : {HF_CACHE_DIR}')
print(f'Output dir : {OUTPUT_DIR}')

## 2. (Colab only) Mount Google Drive

Skip this cell on Kaggle — outputs are saved directly to `/kaggle/working`.

In [ ]:
if PLATFORM == 'colab':
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive mounted at /content/drive')
else:
    print(f'Skipping Drive mount (platform: {PLATFORM})')

## 3. Clone Repository

In [ ]:
if not os.path.exists(REPO_DIR):
    os.system(f'git clone -b cineca https://github.com/AnnikaUnmuessig/Driving_Distraction_Detection.git {REPO_DIR}')
else:
    print(f'Repo already exists at {REPO_DIR}, pulling latest...')
    os.system(f'git -C {REPO_DIR} pull')

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')

## 4. Install Dependencies

In [ ]:
!pip install -q -r requirements.txt
!pip install -q accelerate -U
print('Dependencies installed.')

## 5. Hugging Face Authentication

- **Kaggle**: Set a Secret named `HF_TOKEN` in *Add-ons → Secrets* and enable it for this notebook.
- **Colab**: You will be prompted to enter your token interactively.

In [ ]:
import os
from huggingface_hub import login

if PLATFORM == 'kaggle':
    # Read token from Kaggle Secrets
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret('HF_TOKEN')
    login(token=hf_token, add_to_git_credential=False)
    print('Logged in to Hugging Face via Kaggle Secret.')
else:
    # Colab: check env var first, fall back to interactive login
    hf_token = os.environ.get('HF_TOKEN', '')
    if hf_token:
        login(token=hf_token, add_to_git_credential=False)
        print('Logged in to Hugging Face via HF_TOKEN env variable.')
    else:
        from huggingface_hub import notebook_login
        notebook_login()

## 6. Configuration

Set `VIDEOS_PER_CLASS` to the number of videos to download per class.
Set to `None` to download the full dataset (~13 GB — **not recommended on Kaggle/Colab**).

In [ ]:
# ── USER CONFIGURATION ─────────────────────────────────────────────────────────

# Number of videos to download per class folder.
# Recommended values:
#   50  → fast test (~550 videos total, ~2-3 min download)
#   100 → balanced training (~1100 videos, ~5 min download)
#   160 → same as LIMIT_CAP in Finetuning.py (default)
#   None → full dataset (~13 GB, may take 30+ min)
VIDEOS_PER_CLASS = 100

# Random seed for reproducible file selection
DOWNLOAD_SEED = 42

# Enable Weights & Biases logging (set to False on Kaggle if not configured)
USE_WANDB = False

# ─── (Advanced) override output directory ─────────────────────────────────────
# Uncomment and edit to redirect checkpoints elsewhere:
# OUTPUT_DIR = '/kaggle/working/my_outputs'

# ──────────────────────────────────────────────────────────────────────────────
print(f'Videos per class : {VIDEOS_PER_CLASS if VIDEOS_PER_CLASS else "ALL (full dataset)"}')
print(f'HF cache dir     : {HF_CACHE_DIR}')
print(f'Output dir       : {OUTPUT_DIR}')
print(f'W&B logging      : {USE_WANDB}')

## 7. Download Model & Dataset

Downloads the TimeSformer-HR weights and the selected subset of the distraction dataset.
With `VIDEOS_PER_CLASS = 100` this downloads ~1100 video clips instead of the full ~13 GB.

In [ ]:
# Build the download command
cmd = f'python download_assets.py --output_dir {HF_CACHE_DIR} --seed {DOWNLOAD_SEED}'

if VIDEOS_PER_CLASS is not None:
    cmd += f' --videos_per_class {VIDEOS_PER_CLASS}'

print(f'Running: {cmd}\n')
!{cmd}

## 8. Start Training

Sets environment variables and launches `Finetuning.py`.
Checkpoints and metrics are saved to the output directory.

In [ ]:
import os

os.environ['MODEL_PATH']   = os.path.join(HF_CACHE_DIR, 'timesformer-hr')
os.environ['DATASET_PATH'] = os.path.join(HF_CACHE_DIR, 'distraction_dataset')
os.environ['OUTPUT_DIR']   = OUTPUT_DIR

# Disable W&B if not configured
if not USE_WANDB:
    os.environ['WANDB_MODE']    = 'disabled'
    os.environ['WANDB_DISABLED'] = 'true'

os.makedirs(OUTPUT_DIR, exist_ok=True)

print('Environment:')
print(f'  MODEL_PATH   = {os.environ["MODEL_PATH"]}')
print(f'  DATASET_PATH = {os.environ["DATASET_PATH"]}')
print(f'  OUTPUT_DIR   = {os.environ["OUTPUT_DIR"]}')
print(f'  WANDB_MODE   = {os.environ.get("WANDB_MODE", "online")}')
print()

!python Finetuning.py

## 9. (Kaggle) Save Outputs

On Kaggle, files in `/kaggle/working` are automatically saved as output datasets.
Run this cell to list what was produced.

In [ ]:
import os

output_dir = os.environ.get('OUTPUT_DIR', OUTPUT_DIR)
print(f'Contents of {output_dir}:\n')

if os.path.exists(output_dir):
    for root, dirs, files in os.walk(output_dir):
        level = root.replace(output_dir, '').count(os.sep)
        indent = '  ' * level
        print(f'{indent}{os.path.basename(root)}/')
        sub_indent = '  ' * (level + 1)
        for f in files:
            fpath = os.path.join(root, f)
            size_mb = os.path.getsize(fpath) / (1024 * 1024)
            print(f'{sub_indent}{f}  ({size_mb:.1f} MB)')
else:
    print('Output directory not found. Did training complete successfully?')